# Capítulo 32 · Laboratório 32: as regrinhas da 3ª edição postas à prova
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**: preços ajustados das 40 ações desde 2005, Ibovespa e S&P 500 desde 1995 (Partes 3 e 4, com a limpeza do Capítulo 28) e preços da tela, sem ajuste de proventos e com os desdobramentos desfeitos, desde 2010 (Partes 1 e 2). Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap32.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O sorteio da Parte 4 usa semente fixa. Testes didáticos, em ações que existem hoje (viés de sobrevivência), sem custo: **não são recomendação**.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import re
import time
import numpy as np
import pandas as pd
import yfinance as yf

B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
MERC = {t: ("B3" if t.endswith(".SA") else "EUA") for t in TICKERS}
FIM = "2026-10-01"
rng = np.random.default_rng(34)

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
CACHE = "dados_cap32.pkl"


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def baixa(*args, **kw):
    for tentativa in range(5):                 # o provedor às vezes falha: tenta de novo
        try:
            x = yf.download(*args, progress=False, **kw)
            if len(x):
                return x
        except Exception:
            pass
        time.sleep(5 * (tentativa + 1))
    ativo = args[0] if isinstance(args[0], str) else f"a lista de {len(args[0])} ativos"
    raise RuntimeError(f"Não consegui baixar {ativo} do Yahoo Finance depois de 5 tentativas. O provedor pode estar"
                       " fora do ar ou limitando pedidos. Espere alguns minutos e rode a célula de novo.")


# dados: preços ajustados (Partes 3 e 4); brutos: preços da tela, sem ajuste de proventos (Partes 1 e 2)
if os.path.exists(CACHE):                  # cópia local: o provedor pode corrigir o histórico entre execuções
    dados, ibov, spx, brutos = pd.read_pickle(CACHE)
else:
    dados = baixa(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, group_by="ticker")
    ibov = baixa("^BVSP", start="1995-01-01", end=FIM, auto_adjust=True, multi_level_index=False)
    spx = baixa("^GSPC", start="1995-01-01", end=FIM, auto_adjust=True, multi_level_index=False)
    brutos = baixa(TICKERS, start="2010-01-01", end=FIM, auto_adjust=False, actions=True, group_by="ticker")
    pd.to_pickle((dados, ibov, spx, brutos), CACHE)


def limpa_tela(d):
    """Preços da tela: só tira barras vazias (o desdobramento é desfeito em preco_da_tela)."""
    d = d.dropna(subset=["Open", "High", "Low", "Close"])
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"]) & (d["Volume"] > 0)
    return d[ok]


CORRIGIDAS = {}                            # o que a limpeza mudou (registro da Parte 4; sem repetir)
SUSPEITOS = {"EQTL3.SA": ("2010-08-25", "queda de 40% sem volta (provável ajuste de bonificação ou desdobramento faltando)")}


def limpa(d, nome=""):
    """Limpeza do Capítulo 26, mais três regras para dado sujo que engana o teste de stops:
    1) máxima impossível: pavio de mais de 50% acima da abertura e do fechamento vira o maior dos dois;
    2) desdobramento ou grupamento sem ajuste: salto de mais de 2 vezes (ou menos da metade) da abertura em relação ao
       fechamento anterior, sem volta nos 5 pregões seguintes, corrige o histórico anterior pela mesma proporção;
    3) buraco nos dados: com mais de 15 dias corridos sem cotação, só fica o trecho contínuo depois do último buraco."""
    d = d.dropna().copy()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"])
    d = d[ok & (d["Volume"] > 0)] if "Volume" in d else d[ok]
    px = ["Open", "High", "Low", "Close"]
    topo = d[["Open", "Close"]].max(axis=1)
    alta = d["High"] > 1.5 * topo
    for dt in d.index[alta]:
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: máxima {d.at[dt, 'High']:.2f} com fechamento {d.at[dt, 'Close']:.2f}", None)
    d.loc[alta, "High"] = topo[alta]
    salto = d["Open"] / d["Close"].shift()
    dura = d["Close"].shift(-5) / d["Close"].shift()
    for dt in d.index[((salto < 0.5) | (salto > 2)) & ((dura < 0.6) | (dura > 1.6))]:
        f_ = salto.loc[dt]
        d.loc[d.index < dt, px] *= f_
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: salto de {f_:.4f} vezes sem volta (ajuste que faltou); histórico"
                          " anterior corrigido", None)
    if nome == "SUZB3.SA":                 # antes disso, preço parado no Yahoo (Capítulo 26)
        d = d.loc["2017-11-10":]
    if nome in SUSPEITOS:                  # queda sem volta que parece ajuste faltando, mas sem prova: fica fora
        dt, motivo = SUSPEITOS[nome]
        CORRIGIDAS.setdefault(f"{nome} {pd.Timestamp(dt):%d/%m/%Y}: {motivo}; usa só a partir dessa data", None)
        d = d.loc[dt:]
    buraco = d.index.to_series().diff().dt.days
    if (buraco > 15).any():
        ini = buraco[buraco > 15].index[-1]
        CORRIGIDAS.setdefault(f"{nome}: {int((buraco > 15).sum())} buracos de mais de 15 dias; usa só a partir de {ini:%d/%m/%Y}", None)
        d = d.loc[ini:]
    return d


def preco_da_tela(d):
    """O Yahoo divide o passado pelos desdobramentos. Multiplicar de volta devolve o preço que estava na tela."""
    s = d["Stock Splits"].replace(0, 1.0).fillna(1.0)
    fator = s[::-1].cumprod()[::-1].shift(-1).fillna(1.0)          # produto dos desdobramentos posteriores
    t = d[["Open", "High", "Low", "Close"]].mul(fator, axis=0)
    centavos = (t * 100).round()
    inteiro = ((t * 100 - centavos).abs() < 0.02).all(axis=1)      # só dias com preço em centavos exatos
    return (centavos[inteiro] / 100), inteiro.mean()


def pc(x, casas=1):
    return f"{x:.{casas}%}".replace(".", ",")


TELA, APROV = {}, {}
for tk in TICKERS:
    d = limpa_tela(brutos[tk])
    TELA[tk], APROV[tk] = preco_da_tela(d)

## Parte 1: os centavos preferidos

In [ ]:
print("Parte 1. Últimos dois dígitos (centavos) de máximas, mínimas e fechamentos, de 2010 a 2026")
print(f"   dias com centavos exatos depois de desfazer os desdobramentos: mediana {np.median(list(APROV.values())):.1%}"
      .replace(".", ","))
CENT = {}
for m in ("B3", "EUA"):
    v = np.concatenate([(TELA[t][c] * 100).round().astype(int).to_numpy() % 100
                        for t in TICKERS if MERC[t] == m for c in ("High", "Low", "Close")])
    CENT[m] = pd.Series(v).value_counts(normalize=True).reindex(range(100), fill_value=0)
    f = CENT[m]
    dez = f[[k for k in range(10, 100, 10) if k != 50]].mean()
    cinco = f[[k for k in range(5, 100, 10)]].mean()
    resto = f[[k for k in range(100) if k % 5]].mean()
    print(f"   {m}: {len(v):,} preços".replace(",", ".") + f" | ,00 = {pc(f[0], 2)} | ,50 = {pc(f[50], 2)} |"
          f" outros ,X0 = {pc(dez, 2)} | ,X5 = {pc(cinco, 2)} | demais = {pc(resto, 2)} | ,99 = {pc(f[99], 2)} |"
          f" ,01 = {pc(f[1], 2)}  (acaso: 1,00% cada)")
    for col in ("High", "Low", "Close"):
        w = np.concatenate([(TELA[t][col] * 100).round().astype(int).to_numpy() % 100 for t in TICKERS if MERC[t] == m])
        print(f"      {col:5s}: ,00 = {(w == 0).mean():.2%}  ,50 = {(w == 50).mean():.2%}".replace(".", ","))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 3.5))
plt.bar(range(100), CENT["B3"] * 100, color="gray")
plt.axhline(1, ls=":", color="black")
plt.xlabel("centavos (B3)")
plt.ylabel("% dos preços")
plt.show()

## Parte 2: número redondo segura o preço?

In [ ]:
# Nível de teste = o primeiro preço da forma k + f acima do fechamento anterior (resistência) ou abaixo (suporte),
# com f = 0,00 (número redondo), 0,50 (meio) e 0,10, 0,20... (quebrados, para comparar).
# "Testou" = a máxima chegou ao nível com a abertura abaixo dele. "Rejeitou" = fechou abaixo do nível.
FRACS = [round(x, 2) for x in np.arange(0, 1, 0.1)]


def teste_niveis(p, lado):
    o, h, l, c = p["Open"], p["High"], p["Low"], p["Close"]
    c0 = c.shift()
    linhas = []
    for f in FRACS:
        if lado == "resistência":
            nivel = np.floor(c0 - f + 1e-9) + 1.0 + f                 # primeiro k + f acima de c0
            ok = (h >= nivel - 1e-9) & (o < nivel - 1e-9)
            rej = c < nivel - 1e-9
        else:
            nivel = np.ceil(c0 - f - 1e-9) - 1.0 + f                  # primeiro k + f abaixo de c0
            ok = (l <= nivel + 1e-9) & (o > nivel + 1e-9)
            rej = c > nivel + 1e-9
        faixa = (c0 >= 10) & (c0 <= 60)                               # degrau de R$ 1 / US$ 1 entre 1,7% e 10%
        sel = ok & faixa
        ret5 = c.shift(-5) / c - 1
        passou = sel & ~rej
        sinal = 1 if lado == "resistência" else -1
        linhas.append({"fração": f, "testes": int(sel.sum()), "rejeição": rej[sel].mean(),
                       "5 dias depois de passar": sinal * ret5[passou].mean()})
    return pd.DataFrame(linhas).set_index("fração")


REDONDO = {}
for m in ("B3", "EUA"):
    for lado in ("resistência", "suporte"):
        tabs = [teste_niveis(TELA[t], lado) for t in TICKERS if MERC[t] == m]
        tot = sum(x["testes"] for x in tabs)
        rej = sum(x["rejeição"].fillna(0) * x["testes"] for x in tabs) / tot
        dep = pd.concat([x["5 dias depois de passar"] for x in tabs], axis=1).median(axis=1)
        REDONDO[(m, lado)] = pd.DataFrame({"testes": tot, "rejeição": rej, "5d após passar (mediana)": dep})
print("\nParte 2. Número redondo como suporte e resistência (ações entre 10 e 60 na moeda local, de 2010 a 2026)")
for (m, lado), t in REDONDO.items():
    quebr = t.drop([0.0, 0.5])
    r0, r5, rq = t.loc[0.0, "rejeição"], t.loc[0.5, "rejeição"], quebr["rejeição"].mean()
    n0, nq = t.loc[0.0, "testes"], quebr["testes"].mean()
    # erro-padrão da diferença de proporções (testes tratados como independentes; é otimista)
    ep = np.sqrt(r0 * (1 - r0) / n0 + rq * (1 - rq) / (nq * len(quebr)))
    dp0, dpq = t.loc[0.0, "5d após passar (mediana)"], quebr["5d após passar (mediana)"].mean()
    nn = f"{n0:,}".replace(",", ".")
    print(f"   {m} {lado:12s}: rejeição no ,00 = {pc(r0)} (n = {nn}) | no ,50 = {pc(r5)} | nos quebrados = {pc(rq)}"
          f" (faixa {pc(quebr['rejeição'].min())} a {pc(quebr['rejeição'].max())}) |"
          f" diferença {pc(r0 - rq)} ± {pc(2 * ep)} | 5 dias depois de passar: ,00 {pc(dp0, 2)} x quebrados {pc(dpq, 2)}")

## Parte 3: topos com violência, fundos em silêncio?

In [ ]:
# zigue-zague: topos e fundos confirmados (usado nas Partes 3 e 4)
def zigzag(c, pct=0.20):
    """Topos e fundos confirmados depois de uma volta de pct. Devolve lista de (data, preço, 'topo'/'fundo')."""
    v = c.to_numpy()
    idx = c.index
    pivos, tend, ext_i = [], 0, 0
    for i in range(1, len(v)):
        if tend >= 0 and v[i] > v[ext_i] and tend == 1:
            ext_i = i
        elif tend <= 0 and v[i] < v[ext_i] and tend == -1:
            ext_i = i
        if tend == 0:
            if v[i] >= v[0] * (1 + pct):
                tend, ext_i = 1, i
                pivos.append((0, "fundo"))
            elif v[i] <= v[0] * (1 - pct):
                tend, ext_i = -1, i
                pivos.append((0, "topo"))
        elif tend == 1 and v[i] <= v[ext_i] * (1 - pct):
            pivos.append((ext_i, "topo"))
            tend, ext_i = -1, i
        elif tend == -1 and v[i] >= v[ext_i] * (1 + pct):
            pivos.append((ext_i, "fundo"))
            tend, ext_i = 1, i
    return [(idx[i], v[i], k) for i, k in pivos[1:]]                   # o primeiro ponto é arbitrário


SERIES = {"Ibovespa": ibov["Close"].loc["1995":].dropna(), "S&P 500": spx["Close"].loc["1995":].dropna()}
for t in TICKERS:
    SERIES[t] = limpa(dados[t], t)["Close"]

JAN = 20
linhas, caminhos = [], {"topo": [], "fundo": []}
for nome, c in SERIES.items():
    r = np.log(c).diff()
    pos = {d: i for i, d in enumerate(c.index)}
    for d, p, k in zigzag(c, 0.20):
        i = pos[d]
        if i < 2 * JAN or i + 2 * JAN >= len(c):
            continue
        antes, depois = r.iloc[i - JAN + 1:i + 1], r.iloc[i + 1:i + JAN + 1]
        linhas.append({"série": nome, "tipo": k, "vol antes": antes.std() * np.sqrt(252),
                       "vol depois": depois.std() * np.sqrt(252),
                       "dias de 3%+ (40 dias)": int((pd.concat([antes, depois]).abs() >= 0.03).sum()),
                       "variação nos 20 dias antes": abs(np.exp(antes.sum()) - 1),
                       "variação nos 20 dias depois": abs(np.exp(depois.sum()) - 1),
                       "índice": nome in ("Ibovespa", "S&P 500")})
        seg = c.iloc[i - 2 * JAN:i + 2 * JAN + 1].to_numpy() / p
        caminhos[k].append(seg)
PIV = pd.DataFrame(linhas)
print("\nParte 3. Topos e fundos de 20% (Ibovespa e S&P desde 1995; 40 ações desde 2005): medianas")
for grupo, g in (("todas as séries", PIV), ("só os dois índices", PIV[PIV["índice"]])):
    tab = g.groupby("tipo")[["vol antes", "vol depois", "dias de 3%+ (40 dias)",
                             "variação nos 20 dias antes", "variação nos 20 dias depois"]].median()
    tab.insert(0, "n", g.groupby("tipo").size())
    print(f"   {grupo}:")
    fm = {c: (lambda x: f"{x:.0f}") if c.startswith(("n", "dias")) else (lambda x: pc(x)) for c in tab.columns}
    print(tab.to_string(formatters=fm))
vt = PIV[PIV["tipo"] == "topo"]["vol antes"].to_numpy()
vf = PIV[PIV["tipo"] == "fundo"]["vol antes"].to_numpy()
print(f"   fundos com volatilidade nos 20 dias antes maior que a mediana dos topos: {(vf > np.median(vt)).mean():.0%}"
      .replace(".", ","))

## Parte 4: os últimos 10% do tempo dão 50% da alta?

In [ ]:
def fatia_final(c, pct=0.20, minimo=20):
    """Para cada perna de alta (fundo -> topo) com pelo menos `minimo` pregões:
    parte da alta (em log) que veio nos últimos 10% dos pregões da perna."""
    piv = zigzag(c, pct)
    lc = np.log(c)
    out = []
    for (d0, p0, k0), (d1, p1, k1) in zip(piv[:-1], piv[1:]):
        if k0 != "fundo" or k1 != "topo":
            continue
        perna = lc.loc[d0:d1]
        n = len(perna) - 1
        if n < minimo:
            continue
        corte = int(np.ceil(n * 0.9))
        out.append((perna.iloc[-1] - perna.iloc[corte]) / (perna.iloc[-1] - perna.iloc[0]))
    return out


real, acaso = [], []
for nome, c in SERIES.items():
    real += fatia_final(c)
    r = np.log(c).diff().dropna().to_numpy()
    for _ in range(20):                                              # 20 histórias inventadas por série
        sim = pd.Series(np.exp(np.concatenate([[0], np.cumsum(rng.choice(r, len(r)))])) * c.iloc[0])
        acaso += fatia_final(sim)
real, acaso = np.array(real), np.array(acaso)
print("\nParte 4. Pernas de alta de 20% ou mais (fundo ao topo), parte da alta nos últimos 10% do tempo")
print(f"   real: {len(real)} pernas | mediana {np.median(real):.0%} | média {real.mean():.0%} |"
      f" pernas com 50% ou mais: {(real >= 0.5).mean():.0%}".replace(".", ","))
print(f"   acaso (mesmos retornos sorteados): {len(acaso):_} pernas | mediana {np.median(acaso):.0%} |"
      f" média {acaso.mean():.0%} | pernas com 50% ou mais: {(acaso >= 0.5).mean():.0%}".replace(".", ",").replace("_", "."))
print(f"   o que a limpeza do Capítulo 28 corrigiu nas 40 ações ajustadas ({len(CORRIGIDAS)} casos):")
for linha_ in CORRIGIDAS:
    mostra("      " + linha_)

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `preco_da_tela` da célula de abertura e a contagem dos centavos da Parte 1. Mostre por que o código multiplica de volta os desdobramentos e por que só ficam os dias com preço em centavos exatos." Confira com a Parte 1, que imprime, na B3, ,00 em 2,88% dos preços e ,50 em 2,07%, contra 1,00% do acaso.
2. **Adaptar:** "Numa célula nova, depois da Parte 4, rode `fatia_final` no fechamento de [ATIVO], baixado com a função `baixa` (com `auto_adjust=True` e `multi_level_index=False`) e passado por `limpa`. Não mude o zigue-zague de 20% nem a janela dos últimos 10% do tempo." Antes de olhar o ativo novo, confira se a Parte 4 continua mostrando 522 pernas reais e mediana de 17%, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu caso) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 1, olhe só os fechamentos da B3. Por que o ,00 aparece mais no fechamento que na máxima?
2. Na Parte 2, troque a faixa de preços de 10 a 60 por 5 a 20. O número redondo segura mais o preço?
3. Na Parte 3, troque o zigue-zague de 20% por 10%. O fundo continua mais nervoso que o topo?
4. Na Parte 4, troque os últimos 10% do tempo por 20%. Que parte da alta vem nesse trecho?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.